# Portfolio inequality workshop

Retain 4 layers / scale 1.4 / penalty 1.3 / 20 calls and 3 layers / scale 10 / penalty 30.5 / 20 calls. All assets remain unsigned 2-bit values and the numeric slack is 3 bits (0..7), with weights 1/2/4. There are 512 joint assignments. Cost must include slack before any marginalization; sum(assets)<=6 and sum(assets)+slack==6 are distinct diagnostics.

Preserve every original scenario and problem size. Users supply high-level
costs and classical parameters; the native qfunc prepares uniform registers,
applies positive cost phases, and mixes with RX(beta). No physical wires or bit
decoding are required in the workflow. The companion portfolio_programs.py shows
the complete native quantum functions.

The source uses SciPy COBYLA, whose maxiter limits objective evaluations. Native
COBYLA counts trust-region iterations, so this workflow stops BEFORE an extra
objective execution at the source cap, selects the best actually evaluated
parameters, and reports evaluation_budget and nonconvergence. It does not invent
an optimizer return value or claim the same optimization trajectory. Exact local
mean probabilities replace provider shot-noisy estimates; final 1,000 seeded
samples are actual draws. Sampling/readout and independent verification are
outside the objective-call cap. Classical exhaustive minima are comparison labels,
never substituted for quantum observations. Penalties stay soft; results retain
infeasible assignments. No remote service, hardware or advantage claim.


## Portfolio Optimization with the Quantum Approximate Optimization Algorithm (QAOA)

Native counterpart of original code cell 3.


In [ ]:
from io import BytesIO

from IPython.display import Image, display
from matplotlib import pyplot as plt
from matplotlib.figure import Figure

from examples.synthesis.portfolio_reference import COVARIANCE, RETURNS
from examples.synthesis.tutorials.workshops.finance_workshops.portfolio_programs import (
    slack_program,
)
from examples.synthesis.tutorials.workshops.finance_workshops.portfolio_workflow import (
    INEQUALITY_SCENARIOS,
    assignments,
    bindings,
    build_program,
    classical_optima,
    cost,
    distribution,
    expected_return,
    initial_parameters,
    mean_cost,
    plot_samples,
    plot_trace,
    risk,
    run_workflow,
    verify,
)
from flagquantum.drawer import draw


def show_figure(figure: Figure) -> None:
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=130)
    display(Image(buffer.getvalue()))
    plt.close(figure)


## The Portfolio Optimization Problem Parameters

Native counterpart of original code cell 6.


In [ ]:
print("Expected returns:", RETURNS)
print("Covariance:", COVARIANCE)
total_budget = 6


## Defining the Variables

Native counterpart of original code cell 8.


In [ ]:
scenario = INEQUALITY_SCENARIOS[0]
print("Three two-bit assets and three-bit numeric slack:", len(assignments(scenario)))


## Define the Expected Return

Native counterpart of original code cell 10.


In [ ]:
print("Example expected return:", expected_return((1, 2, 1)))


## Define the Risk Term

Native counterpart of original code cell 12.


In [ ]:
print("Example risk:", risk((1, 2, 1)))


## Define the Entire Portfolio Optimization Objective Function

Native counterpart of original code cell 14.


In [ ]:
print(scenario)
print("Joint encoded cost:", cost(scenario, (1, 2, 1, 2)))
print("Slack weights:", (1, 2, 4))


## Build the QAOA Circuit

Native counterpart of original code cell 16.


In [ ]:
print("Native qfunc:", slack_program)


## Build the QAOA Circuit

Native counterpart of original code cell 17.


In [ ]:
initial = initial_parameters(scenario)
program = build_program(scenario, initial)


## Synthesizing and Visualizing

Native counterpart of original code cell 19.


In [ ]:
print(program.report["resources"])
print(draw(program.circuit))


## Execution and Post Processing

Native counterpart of original code cell 21.


In [ ]:
shots = 1000
print("Interleaved gamma and beta:", initial)


## Define a Callback Function to Track the Optimization

Native counterpart of original code cell 23.


In [ ]:
print("Record actual objective calls; source cap:", scenario.evaluation_budget)


## Define the Objective Function

Native counterpart of original code cell 25.


In [ ]:
probabilities = distribution(scenario, program)
print("Mean over joint assets AND slack:", mean_cost(scenario, probabilities))
assert all(len(a) == 4 for a in probabilities)


## Optimize

Native counterpart of original code cell 27.


In [ ]:
intro = run_workflow(scenario, shots=shots, seed=1950)
print(intro.termination, intro.converged, len(intro.trace))


## Look at the Results

Native counterpart of original code cell 29.


In [ ]:
for row in intro.samples:
    print(row)
print("Budget feasibility and zero penalty are separate columns.")
print("Classical joint objective minima:", classical_optima(scenario))


## Convergence Graph

Native counterpart of original code cell 31.


In [ ]:
show_figure(plot_trace(intro))
show_figure(plot_samples(intro))


## Solution

Native counterpart of original code cell 33.


In [ ]:
inequality = run_workflow(INEQUALITY_SCENARIOS[1], seed=1951)
print(inequality.scenario)
print(inequality.program.report["resources"])
print(inequality.termination, inequality.converged)
print("Actual objective evaluations:", len(inequality.trace))
for row in inequality.samples:
    print(row)
print("Independent classical objective minima:", classical_optima(inequality.scenario))
show_figure(plot_trace(inequality))
show_figure(plot_samples(inequality))
workflows = (
    intro,
    inequality,
)
checks = [
    verify(w.scenario, parameters)
    for w in workflows
    for parameters in (*bindings(w.scenario), w.selected.parameters)
]
assert all(check.maximum_state_error < 3e-11 for check in checks)
assert all(check.dirty_workspace_probability < 1e-22 for check in checks)
for workflow in workflows:
    print(workflow.scenario.name, workflow.termination, workflow.converged)
    print("Actual objective evaluations:", len(workflow.trace))
    print("Initial and selected mean:", workflow.trace[0].cost, workflow.selected.cost)
    print(
        "Best observed (not overwritten by classical optimum):", workflow.best_observed
    )
    print("Independent classical minima:", classical_optima(workflow.scenario))
    assert len(workflow.trace) <= workflow.scenario.evaluation_budget
    assert sum(row.count for row in workflow.samples) == 1000
